# P5 – Control de Trayectorias — MyCobot 280

**Objetivos:**
1. Definir las poses clave: `init_pose`, `watch_pose`, `pick_pose`, `place_pose`
2. Implementar el ciclo de agarre con `send_angles` / `send_coords` + `set_gripper_value`
3. Calibrar velocidades y tiempos de espera
4. Ejecutar 5 ciclos consecutivos sin intervención humana y registrar tasa de éxito
5. Gestionar errores de comunicación con reintentos y timeouts

## 0. Conexión al robot

In [ ]:
import time
import numpy as np
from datetime import datetime
from pymycobot.mycobot import MyCobot

PORT = '/dev/ttyUSB0'
BAUD = 1000000

mc = MyCobot(PORT, BAUD)
mc.power_on()
time.sleep(1)

print(f'Conectado en {PORT} a {BAUD} baud')
print(f'Versión firmware: {mc.get_system_version()}')

## 1. Poses clave del ciclo

Cada pose es un vector `[q1, q2, q3, q4, q5, q6]` en grados (ángulos de hardware).

| Pose | Descripción |
|------|-------------|
| `init_pose` | Posición de reposo segura al inicio y fin de cada ciclo |
| `watch_pose` | Cámara apuntando a la zona de trabajo para detectar el objeto |
| `pick_pose` | Sobre el objeto antes de bajar a agarrar |
| `place_pose` | Zona de depósito donde se suelta el objeto |

In [ ]:
# ---------------------------------------------------------------
# Sección 1: Poses clave (ángulos en grados)
# Calibrar moviendo el robot con la app y leyendo mc.get_angles()
# ---------------------------------------------------------------
POSES = {
    # Todos los joints en 0, J6=-45 orienta la cámara hacia abajo
    'init_pose':  [  0,   0,   0,   0,   0, -45],
    # Cámara apunta a la mesa de trabajo
    'watch_pose': [ 42,   0,   0, -85,  -7,  -3],
    # Pre-agarre: sobre el objeto, antes de bajar
    'pick_pose':  [  0, -20,  30, -10,   0, -45],
    # Zona de depósito
    'place_pose': [ 75,  45,  60, -85,   0, -45],
}

# Coordenadas cartesianas para movimientos finos [x, y, z, rx, ry, rz]
COORDS = {
    'pick_z_upper': 170,   # mm — altura de aproximación
    'pick_z_grasp': 115,   # mm — altura de contacto con el objeto
    'place_coords': [150, 150, 130, -175, 0, -45],
}

# Parámetros de velocidad y gripper
MOVE_SPEED    = 40    # 0-100
GRIPPER_OPEN  = 100   # 0-100
GRIPPER_CLOSE = 20
GRIPPER_SPEED = 80

# Mostrar las poses definidas
print('Poses definidas:')
for nombre, angulos in POSES.items():
    print(f'  {nombre:<12}: {angulos}')

### 1.1 Capturar poses desde el robot

Mueve el robot manualmente a la posición deseada y ejecuta la celda para leer los ángulos reales.

In [ ]:
# ---------------------------------------------------------------
# Lee la pose actual y la guarda en el diccionario POSES
# Cambia POSE_A_CAPTURAR al nombre que quieras calibrar
# ---------------------------------------------------------------
POSE_A_CAPTURAR = 'watch_pose'   # <-- cambiar según necesidad

time.sleep(0.3)
angulos_actuales = mc.get_angles()
coords_actuales  = mc.get_coords()
time.sleep(0.2)

POSES[POSE_A_CAPTURAR] = list(angulos_actuales)

print(f'Pose capturada -> {POSE_A_CAPTURAR}')
print(f'  Ángulos : {[round(a, 2) for a in angulos_actuales]}')
if coords_actuales:
    print(f'  Coords  : x={coords_actuales[0]:.1f}  y={coords_actuales[1]:.1f}  z={coords_actuales[2]:.1f} mm')

## 2. Funciones de movimiento con manejo de errores

Sección 5 del enunciado: reintentos y timeouts ante fallos de comunicación USB/serial.

In [ ]:
# ---------------------------------------------------------------
# Sección 5: Gestión de errores de comunicación
# ---------------------------------------------------------------
MAX_RETRIES  = 3
RETRY_DELAY  = 1.0   # segundos entre reintentos

def send_with_retry(func, *args, retries=MAX_RETRIES):
    """
    Ejecuta func(*args) con reintentos ante excepciones de comunicación.
    Lanza RuntimeError si todos los intentos fallan.
    """
    for intento in range(1, retries + 1):
        try:
            return func(*args)
        except Exception as e:
            print(f'  [REINTENTO {intento}/{retries}] {e}')
            time.sleep(RETRY_DELAY)
    raise RuntimeError(f'Fallo tras {retries} intentos en {func.__name__}')


def goto_angles(pose_name, speed=MOVE_SPEED, wait=2.5):
    """
    Mueve el robot a una pose predefinida por nombre.
    Retorna True si el movimiento fue ejecutado.
    """
    angulos = POSES.get(pose_name)
    if angulos is None:
        print(f'  [ERROR] Pose "{pose_name}" no encontrada')
        return False
    print(f'  -> goto_angles: {pose_name} {angulos}')
    send_with_retry(mc.send_angles, angulos, speed)
    time.sleep(wait)
    return True


def goto_coords(coords, speed=MOVE_SPEED, mode=1, wait=2.5):
    """
    Mueve el robot a coordenadas cartesianas [x, y, z, rx, ry, rz].
    mode=1: movimiento lineal; mode=0: movimiento de ejes.
    """
    print(f'  -> goto_coords: {[round(c,1) for c in coords]}')
    send_with_retry(mc.send_coords, coords, speed, mode)
    time.sleep(wait)
    return True


def move_single_axis(axis, value, speed=MOVE_SPEED, wait=1.5):
    """
    Mueve un único eje cartesiano (1=X, 2=Y, 3=Z, 4=RX, 5=RY, 6=RZ).
    Útil para bajar/subir en Z sin desviarse en XY.
    """
    send_with_retry(mc.send_coord, axis, value, speed)
    time.sleep(wait)


def open_gripper(wait=1.2):
    send_with_retry(mc.set_gripper_value, GRIPPER_OPEN, GRIPPER_SPEED)
    time.sleep(wait)


def close_gripper(wait=1.5):
    send_with_retry(mc.set_gripper_value, GRIPPER_CLOSE, GRIPPER_SPEED)
    time.sleep(wait)


print('Funciones de movimiento definidas.')

## 3. Calibración de velocidades y tiempos de espera

Ejecuta esta celda para probar distintas velocidades y medir el tiempo real de movimiento.

In [ ]:
# ---------------------------------------------------------------
# Sección 3: Calibración de velocidades
# Prueba init_pose -> watch_pose a distintas velocidades
# ---------------------------------------------------------------
velocidades = [20, 40, 60]

print('=' * 55)
print('CALIBRACIÓN DE VELOCIDADES')
print('=' * 55)
print(f'{"Velocidad":>10}  {"T real (s)":>12}  {"Estado":>10}')
print('-' * 55)

resultados_vel = []

for vel in velocidades:
    mc.send_angles(POSES['init_pose'], 50)
    time.sleep(3.0)

    t0 = time.time()
    try:
        mc.send_angles(POSES['watch_pose'], vel)
        time.sleep(0.5)
        # Esperar hasta que el robot no se mueva (is_moving devuelve 0)
        for _ in range(30):
            if mc.is_moving() == 0:
                break
            time.sleep(0.2)
        t_real = time.time() - t0
        estado = 'OK'
    except Exception as e:
        t_real = time.time() - t0
        estado = f'ERROR: {e}'

    resultados_vel.append((vel, t_real, estado))
    print(f'{vel:>10}  {t_real:>12.2f}  {estado:>10}')

print('=' * 55)

# Elegir velocidad óptima (mayor velocidad con estado OK)
vel_optima = max([v for v, t, e in resultados_vel if e == 'OK'], default=40)
MOVE_SPEED = vel_optima
print(f'\nVelocidad seleccionada: {MOVE_SPEED}')

mc.send_angles(POSES['init_pose'], 50)
time.sleep(3)

## 4. Ciclo de agarre: pick y place

**Secuencia pick:**  
`abrir gripper → ir a upper → bajar Z → cerrar gripper → subir Z`

**Secuencia place:**  
`waypoint seguro → ir a depósito → abrir gripper → subir Z`

In [ ]:
# ---------------------------------------------------------------
# Sección 2: Ciclo de agarre
# ---------------------------------------------------------------
SAFE_WAYPOINT = [0.0, 0.0, -90.0, 95.0, 0.0, -45.0]   # pose de clearance


def pick(x, y, rx=-175.0, ry=0.0, rz=-45.0):
    """
    Secuencia de agarre en la posición (x, y) de la mesa.

    Pasos
    -----
    1. Abrir gripper
    2. Mover a coordenadas upper (z = pick_z_upper)
    3. Bajar solo eje Z a pick_z_grasp
    4. Cerrar gripper
    5. Subir eje Z a pick_z_upper

    Retorna bool
    """
    z_upper = COORDS['pick_z_upper']
    z_grasp = COORDS['pick_z_grasp']
    try:
        print('  [PICK] Abriendo gripper')
        open_gripper(1.0)

        print(f'  [PICK] Moviéndose a upper ({x:.1f}, {y:.1f}, {z_upper})')
        goto_coords([x, y, z_upper, rx, ry, rz], wait=2.5)

        print(f'  [PICK] Bajando a Z={z_grasp}')
        move_single_axis(3, z_grasp, wait=1.5)

        print('  [PICK] Cerrando gripper')
        close_gripper(1.5)

        print(f'  [PICK] Subiendo a Z={z_upper}')
        move_single_axis(3, z_upper, wait=2.0)

        return True
    except Exception as e:
        print(f'  [PICK] ERROR: {e}')
        return False


def place():
    """
    Secuencia de depósito en la zona predefinida.

    Pasos
    -----
    1. Ir a waypoint de clearance
    2. Mover a coordenadas de depósito
    3. Abrir gripper
    4. Subir eje Z

    Retorna bool
    """
    try:
        print('  [PLACE] Waypoint de clearance')
        send_with_retry(mc.send_angles, SAFE_WAYPOINT, MOVE_SPEED)
        time.sleep(2.5)

        print('  [PLACE] Moviéndose a zona de depósito')
        goto_coords(COORDS['place_coords'], wait=3.0)

        print('  [PLACE] Abriendo gripper')
        open_gripper(1.0)

        print(f'  [PLACE] Subiendo a Z={COORDS["pick_z_upper"]}')
        move_single_axis(3, COORDS['pick_z_upper'], wait=1.5)

        return True
    except Exception as e:
        print(f'  [PLACE] ERROR: {e}')
        return False


print('Funciones pick() y place() definidas.')

### 4.1 Prueba unitaria de pick y place (1 ciclo manual)

In [ ]:
# ---------------------------------------------------------------
# Prueba de un solo ciclo antes de los 5 consecutivos
# ---------------------------------------------------------------
PICK_X = 150.0   # mm — posición X del objeto en la mesa
PICK_Y =   0.0   # mm — posición Y del objeto en la mesa

print('=== PRUEBA UNITARIA (1 ciclo) ===')

# Fase init
print('Fase 1: init_pose')
goto_angles('init_pose')

# Fase watch
print('Fase 2: watch_pose')
goto_angles('watch_pose')

# Fase pick
print('Fase 3: pick')
ok_pick = pick(PICK_X, PICK_Y)
print(f'  pick resultado: {"OK" if ok_pick else "FALLO"}')

# Fase place
if ok_pick:
    print('Fase 4: place')
    ok_place = place()
    print(f'  place resultado: {"OK" if ok_place else "FALLO"}')

# Volver a init
goto_angles('init_pose')
print('=== Prueba unitaria completada ===')

## 5. Ejecución de 5 ciclos consecutivos

Registra el resultado de cada fase por ciclo y calcula la tasa de éxito final.

In [ ]:
# ---------------------------------------------------------------
# Sección 4: 5 ciclos consecutivos sin intervención humana
# ---------------------------------------------------------------
N_CICLOS = 5

PICK_X = 150.0
PICK_Y =   0.0

log_ciclos = []   # registro de resultados

print('=' * 65)
print(f'INICIO: {N_CICLOS} ciclos consecutivos — {datetime.now().strftime("%H:%M:%S")}')
print('=' * 65)

for ciclo in range(1, N_CICLOS + 1):
    t_inicio = time.time()
    fases = {'init': False, 'watch': False, 'pick': False, 'place': False}
    exito = False

    print(f'\n--- Ciclo {ciclo}/{N_CICLOS} ---')

    # Fase 1: init
    try:
        fases['init'] = goto_angles('init_pose')
    except Exception as e:
        print(f'  [CICLO {ciclo}] init FALLO: {e}')

    if not fases['init']:
        log_ciclos.append({'ciclo': ciclo, 'exito': False,
                           'tiempo': time.time()-t_inicio, 'fases': fases})
        continue

    # Fase 2: watch
    try:
        fases['watch'] = goto_angles('watch_pose')
    except Exception as e:
        print(f'  [CICLO {ciclo}] watch FALLO: {e}')

    if not fases['watch']:
        log_ciclos.append({'ciclo': ciclo, 'exito': False,
                           'tiempo': time.time()-t_inicio, 'fases': fases})
        continue

    # Fase 3: pick
    try:
        fases['pick'] = pick(PICK_X, PICK_Y)
    except Exception as e:
        print(f'  [CICLO {ciclo}] pick FALLO: {e}')

    if not fases['pick']:
        goto_angles('init_pose')   # recuperar a pose segura
        log_ciclos.append({'ciclo': ciclo, 'exito': False,
                           'tiempo': time.time()-t_inicio, 'fases': fases})
        continue

    # Fase 4: place
    try:
        fases['place'] = place()
    except Exception as e:
        print(f'  [CICLO {ciclo}] place FALLO: {e}')

    # Volver a init al final de cada ciclo
    goto_angles('init_pose')

    exito = all(fases.values())
    t_total = time.time() - t_inicio
    log_ciclos.append({'ciclo': ciclo, 'exito': exito,
                       'tiempo': t_total, 'fases': fases})
    print(f'  Ciclo {ciclo}: {"OK" if exito else "FALLO"}  ({t_total:.1f}s)')

    if not exito:
        print(f'  Esperando 3s antes del siguiente ciclo...')
        time.sleep(3)

print('\n' + '=' * 65)
print('FIN DE CICLOS')
print('=' * 65)

## 6. Reporte de métricas

In [ ]:
# ---------------------------------------------------------------
# Tabla de resultados y tasa de éxito
# ---------------------------------------------------------------
n_ok  = sum(1 for r in log_ciclos if r['exito'])
tasa  = n_ok / len(log_ciclos) * 100 if log_ciclos else 0
tiempos = [r['tiempo'] for r in log_ciclos]
t_prom = sum(tiempos) / len(tiempos) if tiempos else 0

print('=' * 70)
print('REPORTE FINAL — 5 CICLOS — MyCobot 280')
print('=' * 70)
print(f'{'Ciclo':<8} {'Éxito':<8} {'Tiempo(s)':<12} {'Init':<8} {'Watch':<8} {'Pick':<8} {'Place':<8}')
print('-' * 70)
for r in log_ciclos:
    f = r['fases']
    estado = lambda v: 'OK' if v else 'FALLO'
    print(f'{r["ciclo"]:<8} {"SI" if r["exito"] else "NO":<8} '
          f'{r["tiempo"]:<12.1f} '
          f'{estado(f["init"]):<8} {estado(f["watch"]):<8} '
          f'{estado(f["pick"]):<8} {estado(f["place"]):<8}')
print('=' * 70)
print(f'Ciclos exitosos   : {n_ok}/{len(log_ciclos)}  ({tasa:.0f}%)')
print(f'Tiempo promedio   : {t_prom:.1f} s/ciclo')
print()
veredicto = 'APROBADO' if tasa >= 80 else 'REQUIERE AJUSTE'
print(f'Resultado: {veredicto}  (criterio: >= 80%)')
print('=' * 70)